# Study Tutor — Corte 1 Project

**AI Agentic Engineering · Ingeniería de Sistemas · Universidad de Santander**

Upload your class notes as a PDF and the tutor maps its topics, writes practice
questions grounded in the text, and grades your answers point by point.

## Architecture

```
                      student message
                            |
                     [ ORCHESTRATOR ]   routes and delegates; never answers itself
                            |
        +-------------------+-------------------+
        |                   |                   |
  Topic Extractor     Exam Generator         Grader
  reads ALL chunks    RAG on the topic       RAG on question + answer
  (no RAG - needs     + verbatim-quote       + per-criterion
   full coverage)       verification           verdicts
        |                   |                   |
        +-------------------+-------------------+
                            |
              ChromaDB  <-  ingestion (plain code, no LLM)
```

Agents live in this notebook. The deterministic plumbing — PDF reading, cleaning,
chunking, embeddings, vector store, retry policy — lives in `tutor/` where it is
unit-tested. See the README for setup.

## Running it

1. `pip install -r ../requirements.txt`
2. Run the setup cell; it creates `../.env` — paste your key from
   [AI Studio](https://aistudio.google.com/apikey).
3. Put a PDF in `../data/`.
4. **Run All.** Cells run top to bottom with no hidden state.

> Editing `.env` or `tutor/*.py` mid-session is handled: the setup cell enables
> `autoreload` and calls `config.reload()`. Re-run it after any change.


---
## 0. Setup


In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [1]:
# Pick up edits to tutor/*.py without restarting the kernel.
%load_ext autoreload
%autoreload 2

import logging
import sys
from pathlib import Path

# The notebook lives in notebooks/, the package one level up.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from tutor import config   # importing this creates .env from .env.example if missing

# autoreload watches .py files, not .env - settings are re-read explicitly.
config.reload()

# Silence the SDK's automatic-function-calling notice; we use no tools.
logging.getLogger('google_genai.models').setLevel(logging.ERROR)

print('repo root      ', config.ROOT_DIR)
print('chat model     ', config.GEMINI_MODEL)
print('embed model    ', config.GEMINI_EMBED_MODEL, f'({config.EMBED_DIM} dims)')
print('chunk size     ', config.CHUNK_SIZE, 'chars, overlap', config.CHUNK_OVERLAP)
print('local fallback ', 'on' if config.ENABLE_LOCAL_FALLBACK else 'off (Gemini only)')
print('PDFs in data/  ', [p.name for p in config.DATA_DIR.glob('*.pdf')] or 'NONE - add one!')

# Fail here, not 20 cells later with a 404 that blames the model.
try:
    config.validate_models()
except Exception as error:
    print('\nCONFIGURATION PROBLEM:\n')
    print(error)

if config.GOOGLE_API_KEY:
    print('API key        loaded OK')
else:
    print(f'API key        MISSING -> open {config.ENV_FILE} and paste your key')


repo root       C:\Users\ACER\Desktop\AI_Agentic_Engineering_Project
chat model      gemini-3.6-flash
embed model     gemini-embedding-001 (768 dims)
chunk size      1200 chars, overlap 200
local fallback  off (Gemini only)
PDFs in data/   ['Test.pdf']
API key        loaded OK


---
## 1. Ingestion — deliberately not an agent

Reading a PDF, stripping headers, chunking and embedding is deterministic. There is no
judgement call, so an LLM here would only add cost and failure modes. It lives in
`tutor/ingest/` as plain, tested code.

Four decisions that make retrieval work:

| Decision | Why |
|---|---|
| Header/footer removal by position + repetition | A footer on 40 pages gets embedded 40 times and floods every result list. |
| Chunks respect paragraphs, 200-char overlap | A chunk cut mid-sentence embeds half an idea. |
| Chunk id = hash of its content | Re-ingesting overwrites instead of duplicating. |
| `RETRIEVAL_DOCUMENT` vs `RETRIEVAL_QUERY` | The embedding model is asymmetric; using one type for both hurts recall. |


In [2]:
from tutor.errors import TutorError
from tutor.ingest.pipeline import ingest
from tutor.vectorstore import get_collection

collection = get_collection()

if collection.count() > 0:
    print(f'Vector store already holds {collection.count()} chunks - skipping ingestion.')
    print('Call ingest(reset=True) if you replaced the PDFs in data/.')
else:
    try:
        stats = ingest()          # reads every PDF in data/
        for key, value in stats.items():
            print(f'{key:20} {value}')
    except TutorError as error:
        print('Ingestion could not run:\n')
        print(error)


  -> opening ChromaDB ...
  <- opening ChromaDB done in 0.2s
Vector store already holds 13 chunks - skipping ingestion.
Call ingest(reset=True) if you replaced the PDFs in data/.


### Retrieval check

Confirm retrieval finds the right material before building on it. Every agent inherits
whatever this returns.

**Read the scores.** If the top hits for a question you know the PDF answers sit below
~0.4, the chunking is wrong — fix it here.

First run is slow (ChromaDB loads, TLS handshake); later runs reuse both. Each stage
prints as it starts, so a silent pause means stuck, not slow — `python scripts/diagnose.py`
times each stage.


In [ ]:
from tutor.vectorstore import search

# Change this to something you know your own PDF answers.
QUESTION = 'resumen de los puntos principales del documento'

for hit in search(QUESTION, top_k=3):
    meta = hit['metadata']
    print(f"[{hit['score']:.3f}] {meta['source']} p.{meta['page']}")
    print('   ', hit['text'][:280].replace(chr(10), ' '), '...')
    print()


---
## 2. The LLM layer

Every agent calls `tutor.llm.generate()`. One retry policy, one structured-output
contract, one place to swap models.

Failures are classified, not lumped together:

| Failure | Response | Why |
|---|---|---|
| `429` quota | stop retrying | retrying cannot create quota |
| `503` / `504` / timeout | retry with backoff (2, 4, 8, 16s + jitter) | Google's shared models spike and recover |
| `4xx` | stop, wrapped with the API's own message | our config is wrong |
| anything else | re-raise raw | our bug, must be loud |

Falling back on *every* error would be worse than no fallback: a broken prompt would
get a quietly worse answer instead of a stack trace.

The local Qwen/Ollama fallback exists but is **off** (`ENABLE_LOCAL_FALLBACK=false`)
while building. Turn it on before the live demo.

> A `503` is Google's model being busy, not your code. Retries handle it.


In [ ]:
from tutor import llm

print('provider       ', config.LLM_PROVIDER)
print('chat model     ', config.active_chat_model())
print('embed model    ', config.active_embed_model(), f'({config.EMBED_PROVIDER})')
print('local fallback ', 'on' if config.ENABLE_LOCAL_FALLBACK else 'off')

needs_ollama = config.LLM_PROVIDER == 'ollama' or config.ENABLE_LOCAL_FALLBACK
if needs_ollama:
    ready = llm.ollama_available()
    print('ollama         ', 'ready' if ready else f'NOT reachable at {config.OLLAMA_HOST}')
    if not ready:
        print(f'                 run: ollama serve   and   ollama pull {config.OLLAMA_MODEL}')


### Switching provider mid-session

The Gemini free tier allows **20 requests per day per model**. When that runs out the
error says *"retry in 2.5s"* — ignore it, that is the per-minute `RetryInfo` attached to
every 429. A daily quota resets tomorrow.

Three ways out, no restart needed — edit `.env`, then re-run the setup cell:

| `.env` | Effect |
|---|---|
| `GEMINI_MODEL=gemini-3.5-flash-lite` | The quota is **per model**, so another one has its own 20. |
| `LLM_PROVIDER=ollama` | Local only. No Gemini call is attempted at all. |
| `ENABLE_LOCAL_FALLBACK=true` | Stay on Gemini, degrade to local automatically on 429. |

Or flip it from here for the rest of the session:


In [ ]:
# Uncomment to force the local model without touching .env.
# Everything downstream reads config.* at call time, so this takes effect immediately.

# config.LLM_PROVIDER = 'ollama'
# print('now using:', config.active_chat_model())

# Embeddings can move too, but the vector store must be rebuilt: vectors from
# different models are not comparable, and vectorstore refuses to mix them.
# config.EMBED_PROVIDER = 'ollama'
# ingest(reset=True)
pass


### Structured output, end to end

The Pydantic model is passed to `generate()` and used to validate the reply. This is
not "please answer in JSON" — the schema constrains decoding, so an invalid shape
cannot be produced. Every agent below uses this pattern.


In [5]:
from pydantic import BaseModel, Field

from tutor.vectorstore import search


class ChunkSanityCheck(BaseModel):
    """Throwaway schema: proves the plumbing works before we build agents."""

    language: str = Field(description='language of the text, e.g. Spanish')
    main_subject: str = Field(description='what this passage is about, in 5 words or fewer')
    is_exam_worthy: bool = Field(description='true if a question could be written from it')


sample = search('tema principal del documento', top_k=1)[0]

response = llm.generate(
    prompt=f'Analyse this passage from a student document:\n\n{sample["text"]}',
    system='You analyse study material. Answer only with the requested structure.',
    schema=ChunkSanityCheck,
)

print('answered by:', response.provider, '|', response.model)
print(response.parse(ChunkSanityCheck))


  cache hit on all 1 - no API call needed
  -> searching 13 chunks ...
  <- searching 13 chunks done in 0.0s
  -> gemini-3.6-flash ...
     still waiting on gemini-3.6-flash ... 5s
  <- gemini-3.6-flash failed after 9.8s
     Gemini busy (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp) - retry 1/4 in 2.3s
  -> gemini-3.6-flash (attempt 2/5) ...
     still waiting on gemini-3.6-flash (attempt 2/5) ... 5s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 10s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 15s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 20s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 25s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 30s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 35s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 40s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 45s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 50s
     still

### The fallback drill (inactive)


In [6]:
# A safety net nobody tested is not one. Simulates a 429; needs ENABLE_LOCAL_FALLBACK=true.
if config.ENABLE_LOCAL_FALLBACK and llm.ollama_available():
    real_gemini = llm._call_gemini

    class SimulatedRateLimit(Exception):
        code = 429

    def rate_limited(*args, **kwargs):
        raise SimulatedRateLimit('429 RESOURCE_EXHAUSTED (simulated)')

    try:
        llm._call_gemini = rate_limited
        forced = llm.generate('Reply with the single word: alive.', temperature=0.0)
        print('answered by:', forced.provider, '|', forced.model)
    finally:
        llm._call_gemini = real_gemini   # restore even if the cell raises
else:
    print('Local fallback disabled - skipping the drill. This is expected right now.')


Local fallback disabled - skipping the drill. This is expected right now.


---
## 3. Agent 1 — Topic Extractor

Returns a structured map of the document: topics, subtopics, pages. Everything
downstream depends on it.

### Why this agent does NOT use RAG

The instinct is to embed "what are the topics?" and take the top 5. Two problems:
that phrase resembles nothing in particular, so the hits are noise; and topic
extraction needs **coverage**, while retrieval is built to discard most of the
document on purpose. Opposite goals.

So it reads everything, in document order. RAG earns its place in agents 2 and 3,
where the question really is "which passages are about *this*?".

### Map-reduce

13 chunks fit in one prompt; 200 do not. **Map**: extract topics per batch.
**Reduce**: a second call merges duplicates across batches. With one batch the reduce
is skipped — it could only cost quota and paraphrase the answer.

The agents live in `tutor/agents.py`, not in these cells, so the CLI
(`python scripts/chat.py`) runs the same code this notebook explains. Their prompts are
files in `prompts/`. The cells below print them.


In [ ]:
from tutor import agents, prompts

# The agents live in tutor/agents.py so the CLI (scripts/chat.py) can use the same
# code this notebook demonstrates. Their prompts are files in prompts/.
print(agents.TopicMap.model_json_schema()['$defs']['Topic']['properties'].keys())
print()
print(prompts.load('topic_extractor'))


The shared persona (`prompts/system_prompt.txt`) carries scope, refusal rules, language
and citation style; each agent adds only its own job on top. Composing them in
`prompts.system()` means no agent can run ungrounded by accident.


In [ ]:
# Every agent runs on the shared persona + its own role. Composing them means no
# agent can run ungrounded by accident.
print(prompts.system(prompts.load('topic_extractor'))[:600], '...')


### The agent

Batches are sized in **characters, not chunks** — the limit that matters is the context
window. Each passage is labelled `[page N]`; without the marker the model would have to
guess page numbers, and it would.


In [ ]:
import inspect

print(inspect.getsource(agents.extract_topics))


In [ ]:
topic_map = agents.extract_topics()

for topic in topic_map.topics:
    pages = ', '.join(str(p) for p in sorted(set(topic.source_pages)))
    print(f'\n{topic.name}   (p. {pages})')
    print(f'  {topic.summary}')
    for subtopic in topic.subtopics:
        print(f'    - {subtopic}')


### Check the result against your PDF

The exam generator inherits every mistake here:

1. **Page numbers correct?** Open one and confirm.
2. **Anything missing?** An omitted topic will never be examined.
3. **The document's words or the model's?** A paraphrase trains you on the wrong terms.


---
## 4. Agent 2 — Exam Generator

**Here RAG is the right tool**: the question is now "which passages are about this
topic?", and we want a small focused subset.

The query uses the topic name **plus its subtopics**. The name alone retrieves too
narrowly to support four genuinely different questions.

Each question ships with `key_points` — what a correct answer must contain. The
generator had the passage in front of it, so it is the cheapest place to decide that.
The grader then checks against explicit criteria instead of re-deriving them: agents
passing structured state, not prose.

`difficulty` is recall / understanding / application, which tells the model something,
unlike easy/medium/hard.


In [ ]:
print(prompts.load('exam_generator'))
print()
print('Exam question fields:', list(agents.ExamQuestion.model_fields))


In [ ]:
print(inspect.getsource(agents.generate_exam))


### Then: verification

The prompt asks for a verbatim quote. Asking is not enough — a model under pressure to
supply a quote produces something quote-shaped, and it looks convincing.

So `tutor/grounding.py` checks each quote against the passages it came from. Fuzzy
enough to tolerate the accents and line breaks pypdf mangles, strict enough to reject a
sentence assembled from scattered words. Failures are flagged, not shipped.


In [ ]:
print(inspect.getsource(agents.verify_exam))


In [ ]:
# Pick any topic from the map above by index.
TOPIC_INDEX = 0
topic = topic_map.topics[TOPIC_INDEX]
print('Topic:', topic.name, '\n')

exam, hits = agents.generate_exam(topic, n_questions=4)
report = agents.verify_exam(exam, hits)

for q, check in zip(exam.questions, report):
    flag = 'OK  ' if check['grounded'] else 'FLAG'
    print(f"\n[{flag}] Q{check['n']} ({q.difficulty}, p.{q.source_page}, "
          f"quote match {check['score']:.0%})")
    print(' ', q.question)
    print('   key points:', '; '.join(q.key_points))
    if not check['grounded']:
        print('   UNVERIFIED QUOTE:', q.source_quote[:160])

passed = sum(c['grounded'] and c['page_ok'] for c in report)
print(f'\n{passed}/{len(report)} questions traced to the document.')


### Reading the flags

- **85-100%** — grounded.
- **50-85%** — probably a real quote the model tidied up. Check it.
- **below 50%** — not in the document. Discard the question.
- **page not retrieved** — the quote may be real but the citation is wrong.

If most questions flag, the cause is usually upstream: chunks too small to hold a
quotable sentence.


---
## 5. Agent 3 — Grader

"Incorrect" teaches nothing. The student needs to know *which* idea they missed and
*where* it is explained.

**1. Grades against explicit criteria.** It walks the `key_points` and marks each
`covered` / `partial` / `missing`. Checking a list, not forming an impression.

**2. Retrieval runs on the student's answer too.** The question finds where the answer
*should* come from; the student's words find where their claims come from — or reveal
they come from nowhere in the document.

**3. The score is computed in code, never asked for.** A model asked for "a score out
of 10" is not reproducible and often contradicts its own words (*"you missed the main
point… 8/10"*). So the model only judges criteria and `tutor/scoring.py` does the
arithmetic. The score therefore cannot disagree with the feedback.


In [ ]:
print(prompts.load('grader'))
print()
print('Feedback fields:', list(agents.RawFeedback.model_fields), '- note: no score')


In [ ]:
print(inspect.getsource(agents.grade_answer))


In [ ]:
from tutor import scoring

print(inspect.getsource(scoring))


### Three answers

Good, half-right, and confidently wrong. The third is the real test: fluent, plausible,
unsupported by the document. A grader that rewards confidence would pass it.


In [ ]:
question = exam.questions[0]
print('Q:', question.question)
print('Key points:', '; '.join(question.key_points), '\n')

# Built from the document itself, so this works with any PDF:
#  - good    : the reference answer
#  - partial : only its first sentence
#  - wrong   : fluent prose about a DIFFERENT topic - plausible, and not an answer
other = next((t for t in topic_map.topics if t.name != topic.name), topic)
ANSWERS = {
    'good': question.expected_answer,
    'partial': question.expected_answer.split('.')[0] + '.',
    'confidently wrong': other.summary,
}

for label, answer in ANSWERS.items():
    print('\n' + '=' * 70)
    print(f'ANSWER ({label}): {answer[:110]}...')
    print('=' * 70)
    print(agents.format_feedback(agents.grade_answer(question, answer)))


### What to check

- The **confidently wrong** answer must be `INCORRECT` with entries under *Incorrect
  claims*. If not, fix `GRADER_ROLE`, not the score.
- The **partial** answer should mix covered and missing. All-partial means the
  criteria are too vague.
- The percentage must match the `+ ~ -` marks — it is computed from them.


---
## 6. The Orchestrator

Something has to decide which agent a message is for. This layer is defined by what it
does **not** do: it never answers from the document itself. Every grounding guarantee
lives inside the specialists, so an orchestrator that answered directly would be an
unchecked fourth agent.

**Routing is structured output** — a `Literal` enforced during decoding, so an
invalid route cannot be returned. No `if 'examen' in message`.

**Arguments are resolved, never trusted.** The topic name the router extracts is a
model output, so it is matched against the real topic map before reaching retrieval.


In [ ]:
from tutor import orchestrator

print(prompts.load('router'))
print()
print('Route fields:', list(orchestrator.Route.model_fields))


In [ ]:
print(inspect.getsource(orchestrator.resolve_topic))


### Dispatch and memory

`handle()` is a boring table from intent to agent call — all the judgement already
happened.

Memory is compacted **after** handling: sliding window of 6 turns plus a rolling
summary. Structured state (open exam, scores) lives in `StudySession`, never in the
transcript — which is what makes the window safe. It can forget turn 1 without
forgetting the exam.


In [ ]:
print(inspect.getsource(orchestrator.handle))


### Full conversation

Nothing is scripted; every message goes through `handle()`. Watch the `[route: ...]`
tag — message 4 is a bare answer with no keywords and still reaches the grader,
because a question is open.


In [ ]:
from tutor.session import StudySession

session = StudySession(topic_map=topic_map)

# Built from the actual topics, so it works with any document.
SCRIPT = [
    '¿De qué trata este documento?',
    f'Explícame el tema: {topic.name}',
    f'Hazme 2 preguntas de práctica sobre {topic.name}',
    ANSWERS['partial'],
    '¿Cómo voy?',
    '¿Cuál es la capital de Francia?',
]

for message in SCRIPT:
    print('\n' + '=' * 74)
    print('STUDENT:', message)
    print('-' * 74)
    print(orchestrator.handle(message, session))


---
## 7. Try it yourself

Nothing above is scripted to one document. This cell is a real conversation loop —
ask anything about the PDF in `data/`, answer a question when one is open, type
`salir` to stop.

The same loop as a program: `python scripts/chat.py`


In [ ]:
session = StudySession(topic_map=topic_map)

print('Ask about your document. Type "salir" to stop.\n')
while True:
    message = input('> ').strip()
    if not message or message.lower() in {'salir', 'exit', 'quit'}:
        break
    print('\n' + orchestrator.handle(message, session) + '\n')

for name, average in session.weakest_topics():
    print(f'{average:.0%}  {name}')


---
## What this demonstrates

| | |
|---|---|
| **Sub-agents** | One router, three specialists, each with its own prompt, schema and retrieval strategy. |
| **RAG where it belongs** | Exam generator and grader; deliberately not the topic extractor. |
| **Context engineering** | Typed state + sliding window with rolling summary. |
| **Verification** | Quotes checked, ungrounded questions dropped, score derived from visible criteria. |
| **Reusable** | `tutor/` is importable and unit-tested; this notebook and `scripts/chat.py` are two front ends over it. |


---
## Appendix — secret leak check

This notebook is committed **with outputs** so it can be read on GitHub. A stray print
or traceback can therefore put the API key in the repo — `.gitignore` protects `.env`,
not this file.

Run after **Save**, before every commit.


In [ ]:
import json, re

nb_path = Path.cwd() / 'tutor.ipynb'
raw = nb_path.read_text(encoding='utf-8')

problems = []
if config.GOOGLE_API_KEY and config.GOOGLE_API_KEY in raw:
    problems.append('your GOOGLE_API_KEY appears verbatim in the saved notebook')
for match in set(re.findall(r'AIza[0-9A-Za-z_\-]{20,}', raw)):
    problems.append(f'a Google API key pattern appears: {match[:8]}...')

if problems:
    print('DO NOT COMMIT:')
    for problem in problems:
        print(' -', problem)
    print('\nClear the offending cell output (Cell > Current Outputs > Clear), save, re-run this check,')
    print('and rotate the key at aistudio.google.com if it was ever pushed.')
else:
    print('Clean: no API key found in the saved notebook. Safe to commit.')
